In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path("../data/raw/ludhiana_rainfall_reference_2022_2025.csv")

if not data_path.exists():
    data_path = Path("data/raw/ludhiana_rainfall_reference_2022_2025.csv")

analysis_df = pd.read_csv(data_path)
analysis_df["date"] = pd.to_datetime(analysis_df["date"])
analysis_df = analysis_df.sort_values("date").reset_index(drop=True)

print("DATA LOADED FOR ANALYSIS")
print("-" * 35)
print("Rows:", len(analysis_df))
print("Date range:", analysis_df["date"].min().date(), "to", analysis_df["date"].max().date())
print("Missing values:", analysis_df.isna().sum().sum())
print("\nRainfall summary (mm):")
print(analysis_df["observed_rainfall_mm"].describe().round(2).to_string())

print("\nMonthly average rainfall (mm):")
print(
    analysis_df.assign(month=analysis_df["date"].dt.month)
    .groupby("month")["observed_rainfall_mm"]
    .mean()
    .round(2)
    .to_string()
)


In [ ]:

monthly_rainfall = (
    analysis_df.assign(month=analysis_df["date"].dt.month)
    .groupby("month")["observed_rainfall_mm"]
    .mean()
)

plt.figure(figsize=(10, 5))
monthly_rainfall.plot(kind="bar")
plt.title("Average Daily Rainfall by Month — Ludhiana (2022–2025)")
plt.xlabel("Month")
plt.ylabel("Average Rainfall (mm)")
plt.xticks(
    ticks=range(12),
    labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun",
            "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    rotation=0
)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

print("Wettest month:", monthly_rainfall.idxmax())
print(f"Highest monthly average: {monthly_rainfall.max():.2f} mm/day")
print("Driest month:", monthly_rainfall.idxmin())
print(f"Lowest monthly average: {monthly_rainfall.min():.2f} mm/day")


In [ ]:

# STEP 48 — Count rainy and dry days

rain_threshold = 0.1  # Rainfall below 0.1 mm is considered dry

analysis_df["rain_status"] = analysis_df["observed_rainfall_mm"].apply(
    lambda rainfall: "Rainy Day" if rainfall >= rain_threshold else "Dry Day"
)

rain_counts = analysis_df["rain_status"].value_counts()
rain_percentages = (
    analysis_df["rain_status"].value_counts(normalize=True) * 100
)

print("RAINY DAYS VS DRY DAYS")
print("-" * 35)

for status in ["Rainy Day", "Dry Day"]:
    print(
        f"{status}: {rain_counts.get(status, 0)} days "
        f"({rain_percentages.get(status, 0):.2f}%)"
    )

print("\nTotal days:", len(analysis_df))


In [ ]:

# STEP 49 — Visualize rainy and dry days

rain_counts = analysis_df["rain_status"].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(
    rain_counts,
    labels=rain_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Rainy Days vs Dry Days — Ludhiana (2022–2025)")
plt.axis("equal")
plt.tight_layout()
plt.show()


In [ ]:

# STEP 50 — Analyze rainfall intensity

def classify_rainfall(mm):
    if mm < 0.1:
        return "Dry"
    elif mm < 2.5:
        return "Light"
    elif mm < 15:
        return "Moderate"
    elif mm < 64.5:
        return "Heavy"
    else:
        return "Very Heavy"

analysis_df["rainfall_category"] = analysis_df["observed_rainfall_mm"].apply(
    classify_rainfall
)

intensity_counts = (
    analysis_df["rainfall_category"]
    .value_counts()
    .reindex(["Dry", "Light", "Moderate", "Heavy", "Very Heavy"], fill_value=0)
)

print("RAINFALL INTENSITY ANALYSIS")
print("-" * 40)

for category, count in intensity_counts.items():
    print(f"{category}: {count} days ({count / len(analysis_df) * 100:.2f}%)")

print("\nMaximum recorded daily rainfall:",
      analysis_df["observed_rainfall_mm"].max(), "mm")


In [ ]:

# STEP 51 — Visualize rainfall intensity distribution

plt.figure(figsize=(9, 5))

intensity_counts.plot(kind="bar")

plt.title("Rainfall Intensity Distribution — Ludhiana (2022–2025)")
plt.xlabel("Rainfall Category")
plt.ylabel("Number of Days")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:

# STEP 52 — Analyze yearly rainfall trends

analysis_df["year"] = analysis_df["date"].dt.year

yearly_rainfall = analysis_df.groupby("year").agg(
    total_rainfall_mm=("observed_rainfall_mm", "sum"),
    average_daily_rainfall_mm=("observed_rainfall_mm", "mean"),
    rainy_days=("observed_rainfall_mm", lambda values: (values >= 0.1).sum())
)

print("YEARLY RAINFALL ANALYSIS")
print("-" * 45)
print(yearly_rainfall.round(2).to_string())

plt.figure(figsize=(9, 5))
yearly_rainfall["total_rainfall_mm"].plot(kind="bar")

plt.title("Total Annual Rainfall — Ludhiana (2022–2025)")
plt.xlabel("Year")
plt.ylabel("Total Rainfall (mm)")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:

# STEP 53 — Plot daily rainfall over time

plt.figure(figsize=(14, 5))

plt.plot(
    analysis_df["date"],
    analysis_df["observed_rainfall_mm"],
    linewidth=0.8
)

plt.title("Daily Rainfall Over Time — Ludhiana (2022–2025)")
plt.xlabel("Date")
plt.ylabel("Daily Rainfall (mm)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

print("Highest daily rainfall:",
      analysis_df["observed_rainfall_mm"].max(), "mm")

print("Number of days with rainfall >= 20 mm:",
      (analysis_df["observed_rainfall_mm"] >= 20).sum())


In [ ]:

# STEP 54 — Analyze rainfall by season

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

analysis_df["season"] = analysis_df["date"].dt.month.apply(get_season)

season_order = ["Winter", "Summer", "Monsoon", "Post-Monsoon"]

seasonal_rainfall = analysis_df.groupby("season").agg(
    average_daily_rainfall_mm=("observed_rainfall_mm", "mean"),
    total_rainfall_mm=("observed_rainfall_mm", "sum"),
    rainy_days=("observed_rainfall_mm", lambda values: (values >= 0.1).sum())
).reindex(season_order)

print("SEASONAL RAINFALL ANALYSIS")
print("-" * 45)
print(seasonal_rainfall.round(2).to_string())

plt.figure(figsize=(9, 5))
seasonal_rainfall["average_daily_rainfall_mm"].plot(kind="bar")

plt.title("Average Daily Rainfall by Season — Ludhiana (2022–2025)")
plt.xlabel("Season")
plt.ylabel("Average Daily Rainfall (mm)")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:

# STEP 55 — Find the wettest and driest dates

wettest_index = analysis_df["observed_rainfall_mm"].idxmax()
driest_index = analysis_df["observed_rainfall_mm"].idxmin()

print("WETTEST DAY")
print("Date:", analysis_df.loc[wettest_index, "date"].date())
print("Rainfall:", analysis_df.loc[wettest_index, "observed_rainfall_mm"], "mm")

print("\nDRIEST DAY")
print("Date:", analysis_df.loc[driest_index, "date"].date())
print("Rainfall:", analysis_df.loc[driest_index, "observed_rainfall_mm"], "mm")


In [ ]:

# STEP 56 — Find consecutive dry periods

dry_mask = analysis_df["observed_rainfall_mm"] < 0.1
dry_group = (~dry_mask).cumsum()

dry_streaks = (
    analysis_df.loc[dry_mask]
    .assign(streak_group=dry_group[dry_mask].values)
    .groupby("streak_group")
    .agg(
        days=("date", "size"),
        start_date=("date", "min"),
        end_date=("date", "max")
    )
)

if not dry_streaks.empty:
    longest_dry = dry_streaks.loc[dry_streaks["days"].idxmax()]
    print("LONGEST CONSECUTIVE DRY PERIOD")
    print("Days:", int(longest_dry["days"]))
    print("Start:", longest_dry["start_date"].date())
    print("End:", longest_dry["end_date"].date())
else:
    print("No dry days found.")


In [ ]:

# STEP 57 — Calculate rolling 7-day rainfall

analysis_df["rainfall_7day_total_mm"] = (
    analysis_df["observed_rainfall_mm"].rolling(window=7).sum()
)

print("7-DAY RAINFALL ANALYSIS")
print("-" * 35)
print("Average 7-day rainfall:",
      round(analysis_df["rainfall_7day_total_mm"].mean(), 2), "mm")
print("Maximum 7-day rainfall:",
      round(analysis_df["rainfall_7day_total_mm"].max(), 2), "mm")

plt.figure(figsize=(12, 5))
plt.plot(
    analysis_df["date"],
    analysis_df["rainfall_7day_total_mm"],
    linewidth=0.8
)
plt.title("Rolling 7-Day Rainfall — Ludhiana (2022–2025)")
plt.xlabel("Date")
plt.ylabel("Rainfall over 7 days (mm)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:

# STEP 58 — Check data quality

analysis_df = analysis_df.sort_values("date").reset_index(drop=True)

print("DATA QUALITY REPORT")
print("-" * 35)
print("Total rows:", len(analysis_df))
print("Missing values:", int(analysis_df.isna().sum().sum()))
print("Duplicate dates:", int(analysis_df["date"].duplicated().sum()))
print("Negative rainfall values:",
      int((analysis_df["observed_rainfall_mm"] < 0).sum()))

date_gaps = analysis_df["date"].diff().dt.days
print("Gaps greater than one day:",
      int((date_gaps > 1).sum()))

print("First date:", analysis_df["date"].min().date())
print("Last date:", analysis_df["date"].max().date())


In [ ]:

# STEP 59 — Monthly rainfall summary

analysis_df["year_month"] = analysis_df["date"].dt.to_period("M")

monthly_summary = analysis_df.groupby("year_month").agg(
    total_rainfall_mm=("observed_rainfall_mm", "sum"),
    average_daily_rainfall_mm=("observed_rainfall_mm", "mean"),
    rainy_days=("observed_rainfall_mm", lambda values: (values >= 0.1).sum()),
    maximum_daily_rainfall_mm=("observed_rainfall_mm", "max")
)

print("MONTHLY SUMMARY — FIRST 12 MONTHS")
print(monthly_summary.head(12).round(2).to_string())

print("\nWettest month in the dataset:")
print(monthly_summary["total_rainfall_mm"].idxmax())

print("Highest monthly total:",
      round(monthly_summary["total_rainfall_mm"].max(), 2), "mm")


In [ ]:

# STEP 60 — Save analysis outputs

from pathlib import Path

output_dir = Path("../data/processed")
if not output_dir.exists():
    output_dir = Path("data/processed")

output_dir.mkdir(parents=True, exist_ok=True)

monthly_summary.to_csv(output_dir / "ludhiana_monthly_rainfall_summary.csv")

seasonal_rainfall.to_csv(output_dir / "ludhiana_seasonal_rainfall_summary.csv")

analysis_df.to_csv(
    output_dir / "ludhiana_rainfall_analysis_2022_2025.csv",
    index=False
)

print("ANALYSIS FILES SAVED SUCCESSFULLY")
print("-" * 40)
print("Folder:", output_dir.resolve())
print("1. ludhiana_monthly_rainfall_summary.csv")
print("2. ludhiana_seasonal_rainfall_summary.csv")
print("3. ludhiana_rainfall_analysis_2022_2025.csv")


## Punjab-wide district analysis

The cells above are the original Ludhiana-only case study. This section analyzes the downloaded ERA5 rainfall dataset for one representative grid point near each of Punjab's 23 district headquarters. These gridded estimates are not district-wide gauge observations. Run `python scripts/download_punjab_district_rainfall.py` from the repository root to refresh the source CSV.

In [ ]:
from pathlib import Path

dataset_path = Path("data/raw/punjab_district_rainfall_2022_2025.csv")
if not dataset_path.exists():
    dataset_path = Path("../data/raw/punjab_district_rainfall_2022_2025.csv")

punjab_df = pd.read_csv(dataset_path, parse_dates=["date"])
punjab_df = punjab_df.sort_values(["district", "date"]).reset_index(drop=True)
punjab_df["year"] = punjab_df["date"].dt.year
punjab_df["month"] = punjab_df["date"].dt.month
punjab_df["rainy_day"] = punjab_df["rainfall_mm"] >= 0.1

district_coverage = punjab_df.groupby("district").agg(
    first_date=("date", "min"),
    last_date=("date", "max"),
    days=("date", "count"),
)
assert len(district_coverage) == 23, f"Expected 23 districts, got {len(district_coverage)}"
assert district_coverage["days"].eq(1461).all(), "Each district should have all 1,461 dates from 2022 through 2025"
assert not punjab_df.duplicated(["district", "date"]).any(), "District-date records should be unique"
assert punjab_df["rainfall_mm"].notna().all() and punjab_df["rainfall_mm"].ge(0).all()

district_summary = punjab_df.groupby("district").agg(
    total_rainfall_mm=("rainfall_mm", "sum"),
    mean_daily_rainfall_mm=("rainfall_mm", "mean"),
    rainy_days=("rainy_day", "sum"),
    observed_days=("rainfall_mm", "count"),
)
district_summary["rainy_day_frequency_pct"] = (
    district_summary["rainy_days"] / district_summary["observed_days"] * 100
)
district_summary = district_summary.sort_values("total_rainfall_mm", ascending=False)

monthly_summary_punjab = punjab_df.groupby(["district", "year", "month"], as_index=False).agg(
    total_rainfall_mm=("rainfall_mm", "sum"),
    mean_daily_rainfall_mm=("rainfall_mm", "mean"),
    rainy_days=("rainy_day", "sum"),
    observed_days=("rainfall_mm", "count"),
)

yearly_summary_punjab = punjab_df.groupby(["district", "year"], as_index=False).agg(
    total_rainfall_mm=("rainfall_mm", "sum"),
    rainy_days=("rainy_day", "sum"),
    observed_days=("rainfall_mm", "count"),
)

longest_dry_streaks = []
for district, district_df in punjab_df.groupby("district", sort=True):
    dry_days = district_df["rainfall_mm"].lt(0.1)
    streak_groups = (~dry_days).cumsum()
    longest_streak = dry_days.groupby(streak_groups).sum().max()
    longest_dry_streaks.append({"district": district, "longest_dry_streak_days": int(longest_streak)})
longest_dry_streaks = pd.DataFrame(longest_dry_streaks).sort_values(
    "longest_dry_streak_days", ascending=False
)

output_dir = Path("data/processed")
if not output_dir.exists():
    output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
district_summary.to_csv(output_dir / "punjab_district_rainfall_summary_2022_2025.csv")
monthly_summary_punjab.to_csv(output_dir / "punjab_district_monthly_rainfall_2022_2025.csv", index=False)
yearly_summary_punjab.to_csv(output_dir / "punjab_district_yearly_rainfall_2022_2025.csv", index=False)
longest_dry_streaks.to_csv(output_dir / "punjab_district_longest_dry_streaks_2022_2025.csv", index=False)

print(f"Loaded {len(punjab_df):,} daily records for {len(district_coverage)} districts.")
print(f"Date range: {punjab_df['date'].min().date()} to {punjab_df['date'].max().date()}")
print("Districts ranked by total rainfall (mm):")
display(district_summary.round(2))
display(longest_dry_streaks.head(10))

district_summary["total_rainfall_mm"].sort_values().plot(
    kind="barh", figsize=(10, 8), title="Punjab District Rainfall, 2022-2025"
)
plt.xlabel("Total rainfall (mm)")
plt.ylabel("District")
plt.tight_layout()
plt.show()

monthly_summary_punjab.head()